# Stage 5: Packaging & Inference

This notebook demonstrates:
1. Single-image and batch-folder prediction via `CatDogPredictor`
2. Robust input handling (grayscale, RGBA, corrupt files)
3. TFLite export and format comparison
4. Latency benchmarking across .keras and .tflite formats

All logic lives in `src/catdog/predict.py` and `src/catdog/export.py`.

In [ ]:
import sys
sys.path.insert(0, '../src')

from pathlib import Path
from catdog.predict import CatDogPredictor, ImageValidationError
from catdog.config import Config

config = Config()
print(f'Project root: {config.project_root}')
print(f'Available models: {[p.name for p in config.models_dir.iterdir() if p.suffix in (".keras", ".tflite")]}')

## 1. Single Image Prediction

In [ ]:
predictor = CatDogPredictor()  # auto-selects best model
print(f'Using model: {predictor.model_path.name}')

# Predict a test image
test_img = config.test_dir / '1.jpg'
result = predictor.predict_image(test_img)
print(result)

## 2. Edge Case Handling

In [ ]:
from PIL import Image as PILImage
import tempfile, os

tmp = Path(tempfile.mkdtemp())

# Grayscale
PILImage.new('L', (100, 100), 128).save(tmp / 'gray.png')
print('Grayscale:', predictor.predict_image(tmp / 'gray.png'))

# RGBA
PILImage.new('RGBA', (100, 100), (255, 0, 0, 128)).save(tmp / 'rgba.png')
print('RGBA:', predictor.predict_image(tmp / 'rgba.png'))

# Corrupt file
(tmp / 'bad.jpg').write_text('not a jpeg')
try:
    predictor.predict_image(tmp / 'bad.jpg')
except ImageValidationError as e:
    print(f'Corrupt file caught: {e}')

## 3. Batch Folder Prediction

In [ ]:
results = predictor.predict_folder(config.test_dir)

for r in results[:5]:
    print(f"{r['image']:>10s}  →  {r['label']}  ({r['confidence_pct']:.1f}%, {r['time_ms']:.1f}ms)")
print(f'... and {len(results) - 5} more')

## 4. TFLite Export & Benchmark

In [ ]:
from catdog.export import run_export_and_benchmark
report = run_export_and_benchmark()